In [14]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import numpy as np
import os

In [16]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406],std = [0.229, 0.224, 0.225] )
])
test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406],std = [0.229, 0.224, 0.225])
])



In [17]:
# extracting and transforming the data using ImageFolder

train_dataset = datasets.ImageFolder(
    root = "train",
    transform = train_transform
)
test_dataset = datasets.ImageFolder(
    root = "test",
    transform = test_transform
)
val_dataset = datasets.ImageFolder(
    root = "val",
    transform = val_transform
)



In [18]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = False)

In [19]:
print(train_dataset.classes)
print(test_dataset.classes)
print(len(train_dataset))
print(len(test_dataset))

['Battery', 'Keyboard', 'Microwave', 'Mobile', 'Mouse', 'PCB', 'Player', 'Printer', 'Television', 'Washing Machine']
['Battery', 'Keyboard', 'Microwave', 'Mobile', 'Mouse', 'PCB', 'Player', 'Printer', 'Television', 'Washing Machine']
2400
300


In [20]:
from torchvision import models
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features  # model.fc means the final fully connected layer
model.fc = nn.Linear(num_features, 10) # in transfer learning smaller classifying layer is used

c:\Users\Hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [21]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

In [22]:
#model = EWasteCNN(input_features=3)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr = 0.0001)

In [23]:
training_log = []

epochs = 20

for epoch in range(epochs):

    model.train()

    correct = 0
    total = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = correct / total


    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_acc = correct / total


    training_log.append({
        "epoch": epoch + 1,
        "train_acc": train_acc,
        "test_acc": test_acc
    })


    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

Epoch 1/20
Train Acc: 0.1829 | Test Acc: 0.3033
Epoch 2/20
Train Acc: 0.3900 | Test Acc: 0.4667
Epoch 3/20
Train Acc: 0.5300 | Test Acc: 0.5900
Epoch 4/20
Train Acc: 0.6408 | Test Acc: 0.6667
Epoch 5/20
Train Acc: 0.6850 | Test Acc: 0.7267
Epoch 6/20
Train Acc: 0.7296 | Test Acc: 0.7533
Epoch 7/20
Train Acc: 0.7608 | Test Acc: 0.7533
Epoch 8/20
Train Acc: 0.7783 | Test Acc: 0.8067
Epoch 9/20
Train Acc: 0.7887 | Test Acc: 0.8033
Epoch 10/20
Train Acc: 0.8033 | Test Acc: 0.8233
Epoch 11/20
Train Acc: 0.8087 | Test Acc: 0.8133
Epoch 12/20
Train Acc: 0.8187 | Test Acc: 0.8167
Epoch 13/20
Train Acc: 0.8196 | Test Acc: 0.8300
Epoch 14/20
Train Acc: 0.8150 | Test Acc: 0.8233
Epoch 15/20
Train Acc: 0.8321 | Test Acc: 0.8267
Epoch 16/20
Train Acc: 0.8354 | Test Acc: 0.8367
Epoch 17/20
Train Acc: 0.8367 | Test Acc: 0.8200
Epoch 18/20
Train Acc: 0.8554 | Test Acc: 0.8300
Epoch 19/20
Train Acc: 0.8442 | Test Acc: 0.8500
Epoch 20/20
Train Acc: 0.8413 | Test Acc: 0.8400


In [24]:
import json

with open("cnn_transfer_learning_metrics.json","w") as f:
    json.dump(training_log, f, indent = 2)


In [32]:
torch.save(model.state_dict(), "ewaste_model_transfer_learning.pth")

In [28]:
model.load_state_dict(torch.load("ewaste_model_transfer_learning.pth"))

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:

        images = images
        labels = labels

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print("Test Accuracy:", accuracy)

Test Accuracy: 84.0


In [31]:
model.load_state_dict(torch.load("ewaste_model_transfer_learning.pth"))

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in train_loader:

        images = images
        labels = labels

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print("Train Accuracy:", accuracy)

Train Accuracy: 86.83333333333333
